# Python Assignment


## 1. Imports

In [3]:
"""
Modules imported:
    - numpy      : numerical computations
    - pandas     : data manipulation / DataFrames
    - unittest   : built-in unit-testing framework
    - logging    : structured log messages
    - os         : file-system checks
    - sklearn    : mean_squared_error metric
    - sqlalchemy : SQL database engine
    - bokeh      : interactive visualisation (optional)
"""

import numpy as np
import pandas as pd
import unittest
import logging
import os
from sklearn.metrics import mean_squared_error
from sqlalchemy import create_engine

# Bokeh is optional – skip gracefully if absent
try:
    from bokeh.plotting import figure, show
    BOKEH_AVAILABLE = True
except ImportError:
    BOKEH_AVAILABLE = False
    print("⚠️  Bokeh not installed – visualisation cells will be skipped.")

## 2. Logging Setup

In [5]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
)
logger = logging.getLogger(__name__)
logger.info("Logger initialised.")

2026-05-12 16:00:36,564 [INFO] Logger initialised.


## 3. Custom Exception Classes
Typed exceptions make error handling explicit and readable.

In [7]:
class DataLoadError(Exception):
    """Raised when a CSV file cannot be loaded."""

class MissingColumnError(Exception):
    """Raised when an expected column is absent from a DataFrame."""

class EmptyDataError(Exception):
    """Raised when a DataFrame has no rows."""

class DatabaseError(Exception):
    """Raised when a database operation fails."""

class MappingError(Exception):
    """Raised when test-data mapping cannot be completed."""

print("✅ Custom exception classes defined.")

✅ Custom exception classes defined.


## 4. Data Loading
`load_csv` validates existence, parse success, non-empty rows, and required columns.

In [9]:
def load_csv(filepath: str, required_columns: list | None = None) -> pd.DataFrame:
    """
    Load a CSV file into a DataFrame with full validation.

    Parameters
    ----------
    filepath         : path to the CSV file
    required_columns : column names that must be present

    Returns
    -------
    pd.DataFrame

    Raises
    ------
    DataLoadError      – file not found or parse error
    EmptyDataError     – file loaded but contains no rows
    MissingColumnError – a required column is absent
    """
    if not os.path.exists(filepath):
        raise DataLoadError(f"File not found: '{filepath}'")

    try:
        df = pd.read_csv(filepath)
    except pd.errors.ParserError as exc:
        raise DataLoadError(f"Could not parse '{filepath}': {exc}") from exc
    except Exception as exc:
        raise DataLoadError(f"Unexpected error loading '{filepath}': {exc}") from exc

    if df.empty:
        raise EmptyDataError(f"'{filepath}' loaded but contains no rows.")

    if required_columns:
        for col in required_columns:
            if col not in df.columns:
                raise MissingColumnError(
                    f"Expected column '{col}' not found in '{filepath}'. "
                    f"Available: {list(df.columns)}"
                )

    logger.info("Loaded '%s' → %d rows × %d cols.", filepath, len(df), len(df.columns))
    return df

### Load training, ideal, and test data

In [11]:
try:
    train_data = load_csv("train.csv",  required_columns=["x","y1","y2","y3","y4"])
    ideal_data = load_csv("ideal.csv",  required_columns=["x"])
    test_data  = load_csv("test.csv",   required_columns=["x","y"])
    print("✅ All CSV files loaded successfully.")
except (DataLoadError, MissingColumnError, EmptyDataError) as exc:
    print(f"❌ Data loading failed: {exc}")

2026-05-12 16:00:36,591 [INFO] Loaded 'train.csv' → 400 rows × 5 cols.
2026-05-12 16:00:36,593 [INFO] Loaded 'ideal.csv' → 400 rows × 51 cols.
2026-05-12 16:00:36,594 [INFO] Loaded 'test.csv' → 100 rows × 2 cols.


✅ All CSV files loaded successfully.


## 5. Exploratory Data Analysis

In [13]:
train_data.head()

,x,y1,y2,y3,y4
0,-20.0,39.778572,-40.078590,-20.214268,-0.324914
1,-19.9,39.604813,-39.784000,-20.070950,-0.058820
2,-19.8,40.099070,-40.018845,-19.906782,-0.451830
3,-19.7,40.151100,-39.518402,-19.389118,-0.612044
4,-19.6,39.795662,-39.360065,-19.815890,-0.306076


In [14]:
train_data.describe()

,x,y1,y2,y3,y4
count,400.00000,400.000000,400.000000,400.000000,400.000000
mean,-0.05000,0.107666,-0.094239,-0.051628,0.012633
std,11.56143,23.103285,23.111794,11.560369,0.327063
min,-20.00000,-39.724934,-40.078590,-20.214268,-0.744510
25%,-10.02500,-20.312566,-19.642560,-9.999807,-0.198240
50%,-0.05000,0.367844,-0.000603,-0.155205,0.024634
75%,9.92500,19.606209,19.478971,9.992209,0.236008
max,19.90000,40.151100,40.558865,19.840752,0.742489


In [15]:
train_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 400 entries, 0 to 399
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   x       400 non-null    float64
 1   y1      400 non-null    float64
 2   y2      400 non-null    float64
 3   y3      400 non-null    float64
 4   y4      400 non-null    float64
dtypes: float64(5)
memory usage: 15.8 KB


## 6. Visualise Training Data

In [17]:
if BOKEH_AVAILABLE:
    p = figure(title="Training Data", x_axis_label="X - Values", y_axis_label="Y - Values")
    colors = {"y1": "red", "y2": "blue", "y3": "green", "y4": "yellow"}
    for col, color in colors.items():
        if col in train_data.columns:
            p.scatter(train_data["x"], train_data[col], size=5,
                      color=color, marker="star", legend_label=col)
    show(p)
else:
    print("Bokeh not available – skipping plot.")

## 7. Find Best-Fit Ideal Functions
For each training column (y1–y4) find the ideal column (y1–y50) with minimum MSE.

In [19]:
def find_best_fit_functions(train_data: pd.DataFrame,
                             ideal_data: pd.DataFrame) -> dict:
    """
    For each training column (y1–y4), find the ideal column (y1–y50)
    that minimises Mean Squared Error.

    Returns
    -------
    dict – e.g. {'y1': 'y42', 'y2': 'y41', ...}

    Raises
    ------
    MissingColumnError – 'x' absent from either DataFrame
    ValueError         – ideal_data has no numeric y-columns
    MappingError       – no suitable match found for a training column
    """
    for df_name, df in [("train_data", train_data), ("ideal_data", ideal_data)]:
        if "x" not in df.columns:
            raise MissingColumnError(f"'x' column missing from {df_name}.")

    train_y_cols = [f"y{i}" for i in range(1, 5)]
    ideal_y_cols = [c for c in ideal_data.columns if c.startswith("y")]

    if not ideal_y_cols:
        raise ValueError("ideal_data has no y-columns to compare against.")

    best_fit_functions: dict = {}

    for y_train in train_y_cols:
        if y_train not in train_data.columns:
            raise MissingColumnError(f"Training column '{y_train}' not found.")

        best_mse = float("inf")
        best_fit = None

        for y_ideal in ideal_y_cols:
            if y_ideal not in ideal_data.columns:
                continue
            try:
                mse = mean_squared_error(train_data[y_train], ideal_data[y_ideal])
            except Exception as exc:
                logger.warning("MSE failed for %s vs %s: %s", y_train, y_ideal, exc)
                continue
            if mse < best_mse:
                best_mse = mse
                best_fit = y_ideal

        if best_fit is None:
            raise MappingError(f"No suitable ideal function for '{y_train}'.")

        best_fit_functions[y_train] = best_fit
        logger.info("%s → %s  (MSE=%.6f)", y_train, best_fit, best_mse)

    return best_fit_functions

In [20]:
try:
    best_fit_functions = find_best_fit_functions(train_data, ideal_data)
    print("Best-fit functions:", best_fit_functions)
except (MissingColumnError, MappingError, ValueError) as exc:
    print(f"❌ {exc}")

2026-05-12 16:00:36,795 [INFO] y1 → y42  (MSE=0.085616)
2026-05-12 16:00:36,806 [INFO] y2 → y41  (MSE=0.089005)
2026-05-12 16:00:36,813 [INFO] y3 → y11  (MSE=0.074655)
2026-05-12 16:00:36,819 [INFO] y4 → y48  (MSE=0.079909)


Best-fit functions: {'y1': 'y42', 'y2': 'y41', 'y3': 'y11', 'y4': 'y48'}


## 8. Visualise Training vs Best-Fit Ideal Functions

In [22]:
if BOKEH_AVAILABLE:
    p = figure(title="Training vs Best-Fit Ideal Functions",
               x_axis_label="X - Values", y_axis_label="Y - Values")

    train_colors = {"y1": "red", "y2": "green", "y3": "black", "y4": "yellow"}
    ideal_colors = {"y42": "violet", "y41": "blue", "y11": "pink", "y48": "grey"}

    for col, color in train_colors.items():
        if col in train_data.columns:
            p.scatter(train_data["x"], train_data[col], size=5, color=color,
                      marker="circle", legend_label=f"train {col}")

    for _, ideal_col in best_fit_functions.items():
        color = ideal_colors.get(ideal_col, "orange")
        if ideal_col in ideal_data.columns:
            p.scatter(ideal_data["x"], ideal_data[ideal_col], size=5, color=color,
                      marker="star", legend_label=f"ideal {ideal_col}")
    show(p)
else:
    print("Bokeh not available – skipping plot.")

## 9. Maximum Deviations
Compute the largest absolute deviation between each training column and its ideal match.
These values form the thresholds (× √2) used when mapping test data.

In [24]:
def find_max_deviations(train_data: pd.DataFrame,
                         ideal_data: pd.DataFrame,
                         best_fit_functions: dict) -> dict:
    """
    Compute max |train_col - ideal_col| for each pair in best_fit_functions.

    Raises
    ------
    MissingColumnError – required column absent
    MappingError       – computation error
    """
    max_deviation: dict = {}
    for y_col, ideal_col in best_fit_functions.items():
        if y_col not in train_data.columns:
            raise MissingColumnError(f"'{y_col}' not in train_data.")
        if ideal_col not in ideal_data.columns:
            raise MissingColumnError(f"'{ideal_col}' not in ideal_data.")
        try:
            max_deviation[y_col] = float(
                np.max(np.abs(train_data[y_col].values - ideal_data[ideal_col].values))
            )
        except Exception as exc:
            raise MappingError(f"Error computing deviation for {y_col}: {exc}") from exc
    return max_deviation

In [25]:
try:
    max_deviation = find_max_deviations(train_data, ideal_data, best_fit_functions)
    print("Max deviations:", max_deviation)
except (MissingColumnError, MappingError) as exc:
    print(f"❌ {exc}")

Max deviations: {'y1': 0.4959680000000013, 'y2': 0.49770300000000134, 'y3': 0.4989360000000005, 'y4': 0.49974158999999996}


## 10. Map Test Data to Ideal Functions
A test point (x, y) is assigned to the best-matching ideal function when
its deviation ≤ √2 × max_deviation(training).

In [27]:
def map_test_data(test_data: pd.DataFrame,
                   ideal_data: pd.DataFrame,
                   best_fit_functions: dict,
                   max_deviation: dict) -> pd.DataFrame:
    """
    Assign each test point to the closest matching ideal function.

    Returns
    -------
    pd.DataFrame with columns [x, y, deviation, chosen_ideal_function]

    Raises
    ------
    MissingColumnError – 'x' or 'y' absent from test_data
    EmptyDataError     – test_data is empty
    """
    for col in ("x", "y"):
        if col not in test_data.columns:
            raise MissingColumnError(f"'{col}' missing from test_data.")
    if test_data.empty:
        raise EmptyDataError("test_data contains no rows.")

    records = []
    for _, row in test_data.iterrows():
        x_test, y_test = row["x"], row["y"]
        chosen_fit, min_dev = None, None

        for y_col, ideal_col in best_fit_functions.items():
            try:
                ideal_vals = ideal_data.loc[ideal_data["x"] == x_test, ideal_col].values
            except Exception as exc:
                logger.warning("Lookup failed x=%s col=%s: %s", x_test, ideal_col, exc)
                continue
            if len(ideal_vals) == 0:
                continue
            deviation = float(np.abs(y_test - ideal_vals[0]))
            if deviation <= max_deviation[y_col] * np.sqrt(2):
                if min_dev is None or deviation < min_dev:
                    min_dev = deviation
                    chosen_fit = ideal_col

        records.append({"x": x_test, "y": y_test,
                         "deviation": min_dev,
                         "chosen_ideal_function": chosen_fit})
    return pd.DataFrame(records)

In [28]:
try:
    mapped_df = map_test_data(test_data, ideal_data, best_fit_functions, max_deviation)
    print("Mapped Data (first 5 rows):")
    display(mapped_df.head())
except (MissingColumnError, EmptyDataError, MappingError) as exc:
    print(f"❌ {exc}")

Mapped Data (first 5 rows):


,x,y,deviation,chosen_ideal_function
0,17.5,34.161040,0.351148,y41
1,0.3,1.215102,0.467342,y41
2,-8.7,-16.843908,NaN,None
3,-19.2,-37.170870,NaN,None
4,-11.0,-20.263054,NaN,None


## 11. Count Mapped Points per Ideal Function

In [30]:
def count_mapped_points(mapped_df: pd.DataFrame) -> pd.DataFrame:
    """
    Return a summary DataFrame of point counts per ideal function.

    Raises
    ------
    MissingColumnError – 'chosen_ideal_function' column absent
    """
    if "chosen_ideal_function" not in mapped_df.columns:
        raise MissingColumnError("'chosen_ideal_function' column not found.")
    counts = (
        mapped_df["chosen_ideal_function"]
        .value_counts(dropna=False)
        .reset_index()
    )
    counts.columns = ["chosen_ideal_function", "count"]
    return counts

In [31]:
try:
    counts_df = count_mapped_points(mapped_df)
    print("Points assigned to each ideal function:")
    display(counts_df)
except MissingColumnError as exc:
    print(f"❌ {exc}")

Points assigned to each ideal function:


,chosen_ideal_function,count
0,None,52
1,y41,13
2,y11,12
3,y42,12
4,y48,11


## 12. Save Data to SQLite Database
All DataFrames are persisted to `assignment.db` using SQLAlchemy.

In [33]:
def get_engine(db_path: str = "assignment.db"):
    """Return a SQLAlchemy engine for the given SQLite path."""
    try:
        engine = create_engine(f"sqlite:///{db_path}")
        logger.info("Database engine created: %s", db_path)
        return engine
    except Exception as exc:
        raise DatabaseError(f"Could not create engine: {exc}") from exc


def save_to_db(df: pd.DataFrame, table_name: str, engine,
               if_exists: str = "replace") -> None:
    """
    Save a DataFrame to a SQLite table.

    Raises
    ------
    DatabaseError – empty DataFrame or write failure
    """
    if df is None or df.empty:
        raise DatabaseError(f"Cannot save empty DataFrame to '{table_name}'.")
    try:
        df.to_sql(table_name, con=engine, if_exists=if_exists, index=False)
        logger.info("Saved %d rows → table '%s'.", len(df), table_name)
    except Exception as exc:
        raise DatabaseError(f"Failed writing to '{table_name}': {exc}") from exc


def load_from_db(table_name: str, engine) -> pd.DataFrame:
    """Read a table from the SQLite database."""
    try:
        df = pd.read_sql_table(table_name, con=engine)
        logger.info("Read %d rows ← table '%s'.", len(df), table_name)
        return df
    except Exception as exc:
        raise DatabaseError(f"Failed reading '{table_name}': {exc}") from exc

In [34]:
try:
    engine = get_engine("assignment.db")
    save_to_db(train_data,  "training_data",  engine)
    save_to_db(ideal_data,  "ideal_data",     engine)
    save_to_db(test_data,   "test_data",      engine)
    save_to_db(mapped_df,   "mapped_results", engine)
    save_to_db(counts_df,   "mapping_counts", engine)
    print("✅ All tables saved to assignment.db")
except DatabaseError as exc:
    print(f"❌ Database error: {exc}")

2026-05-12 16:00:37,067 [INFO] Database engine created: assignment.db
2026-05-12 16:00:37,077 [INFO] Saved 400 rows → table 'training_data'.
2026-05-12 16:00:37,090 [INFO] Saved 400 rows → table 'ideal_data'.
2026-05-12 16:00:37,093 [INFO] Saved 100 rows → table 'test_data'.
2026-05-12 16:00:37,096 [INFO] Saved 100 rows → table 'mapped_results'.
2026-05-12 16:00:37,098 [INFO] Saved 5 rows → table 'mapping_counts'.


✅ All tables saved to assignment.db


### Verify – reload mapped_results from DB

In [36]:
try:
    reloaded = load_from_db("mapped_results", engine)
    print(f"Reloaded {len(reloaded)} rows from 'mapped_results'.")
    display(reloaded.head())
except DatabaseError as exc:
    print(f"❌ {exc}")

2026-05-12 16:00:37,110 [INFO] Read 100 rows ← table 'mapped_results'.


Reloaded 100 rows from 'mapped_results'.


,x,y,deviation,chosen_ideal_function
0,17.5,34.161040,0.351148,y41
1,0.3,1.215102,0.467342,y41
2,-8.7,-16.843908,NaN,None
3,-19.2,-37.170870,NaN,None
4,-11.0,-20.263054,NaN,None


## 13. Unit Tests
Six test classes, 16 test cases covering all core functions.
Run the cell to execute the full suite inline.

In [38]:
class TestDataLoading(unittest.TestCase):

    def test_file_not_found_raises(self):
        with self.assertRaises(DataLoadError):
            load_csv("nonexistent_file.csv")

    def test_missing_required_column_raises(self):
        pd.DataFrame({"a": [1, 2]}).to_csv("/tmp/_tc_missing.csv", index=False)
        with self.assertRaises(MissingColumnError):
            load_csv("/tmp/_tc_missing.csv", required_columns=["x"])

    def test_valid_csv_loads(self):
        pd.DataFrame({"x": [1.0], "y": [2.0]}).to_csv("/tmp/_tc_valid.csv", index=False)
        df = load_csv("/tmp/_tc_valid.csv", required_columns=["x", "y"])
        self.assertEqual(len(df), 1)


class TestFindBestFit(unittest.TestCase):

    def _make_data(self):
        x = np.linspace(-5, 5, 50)
        rng = np.random.default_rng(42)
        train = pd.DataFrame({
            "x": x,
            "y1": 2*x + rng.normal(0, 0.05, 50),
            "y2": np.sin(x) + rng.normal(0, 0.05, 50),
            "y3": x**2 + rng.normal(0, 0.05, 50),
            "y4": np.cos(x) + rng.normal(0, 0.05, 50),
        })
        ideal_cols = {"x": x}
        for i in range(1, 51):
            ideal_cols[f"y{i}"] = rng.uniform(-10, 10, 50)
        ideal_cols["y10"] = 2*x          # best match for y1
        ideal_cols["y20"] = np.sin(x)    # best match for y2
        ideal_cols["y30"] = x**2         # best match for y3
        ideal_cols["y40"] = np.cos(x)    # best match for y4
        return train, pd.DataFrame(ideal_cols)

    def test_returns_four_keys(self):
        t, i = self._make_data()
        result = find_best_fit_functions(t, i)
        self.assertEqual(set(result.keys()), {"y1","y2","y3","y4"})

    def test_values_are_valid_ideal_cols(self):
        t, i = self._make_data()
        result = find_best_fit_functions(t, i)
        valid = {c for c in i.columns if c.startswith("y")}
        for v in result.values():
            self.assertIn(v, valid)

    def test_missing_x_raises(self):
        t = pd.DataFrame({"y1":[1],"y2":[2],"y3":[3],"y4":[4]})
        i = pd.DataFrame({"x":[1],"y1":[1]})
        with self.assertRaises(MissingColumnError):
            find_best_fit_functions(t, i)

    def test_no_ideal_y_cols_raises(self):
        t = pd.DataFrame({"x":[1],"y1":[1],"y2":[2],"y3":[3],"y4":[4]})
        i = pd.DataFrame({"x":[1],"z1":[1]})
        with self.assertRaises(ValueError):
            find_best_fit_functions(t, i)


class TestFindMaxDeviations(unittest.TestCase):

    def test_correct_deviation(self):
        x = np.array([1.,2.,3.])
        train = pd.DataFrame({"x":x,"y1":[1.,2.,3.],"y2":[0.,0.,0.],
                               "y3":[0.,0.,0.],"y4":[0.,0.,0.]})
        ideal = pd.DataFrame({"x":x,"y5":[1.5,2.5,3.5],"y6":[0.,0.,0.],
                               "y7":[0.,0.,0.],"y8":[0.,0.,0.]})
        bff = {"y1":"y5","y2":"y6","y3":"y7","y4":"y8"}
        res = find_max_deviations(train, ideal, bff)
        self.assertAlmostEqual(res["y1"], 0.5)
        self.assertAlmostEqual(res["y2"], 0.0)

    def test_missing_ideal_col_raises(self):
        train = pd.DataFrame({"x":[1],"y1":[1],"y2":[0],"y3":[0],"y4":[0]})
        ideal = pd.DataFrame({"x":[1],"y9":[2]})
        with self.assertRaises(MissingColumnError):
            find_max_deviations(train, ideal, {"y1":"y_missing"})


class TestMapTestData(unittest.TestCase):

    def _setup(self):
        x = np.array([1.,2.,3.,4.,5.])
        ideal = pd.DataFrame({"x":x,"y5":2*x})
        test  = pd.DataFrame({"x":[1.,2.,99.],"y":[2.1,4.05,0.]})
        return test, ideal, {"y1":"y5"}, {"y1":0.2}

    def test_matched_points_have_deviation(self):
        test, ideal, bff, md_ = self._setup()
        res = map_test_data(test, ideal, bff, md_)
        self.assertIsNotNone(res.loc[0,"deviation"])
        self.assertIsNotNone(res.loc[1,"deviation"])

    def test_unmatched_x_is_none(self):
        test, ideal, bff, md_ = self._setup()
        res = map_test_data(test, ideal, bff, md_)
        self.assertIsNone(res.loc[2,"chosen_ideal_function"])

    def test_missing_x_col_raises(self):
        with self.assertRaises(MissingColumnError):
            map_test_data(pd.DataFrame({"y":[1.]}),
                          pd.DataFrame({"x":[1.],"y5":[2.]}),
                          {"y1":"y5"}, {"y1":0.5})

    def test_empty_test_raises(self):
        with self.assertRaises(EmptyDataError):
            map_test_data(pd.DataFrame(columns=["x","y"]),
                          pd.DataFrame({"x":[1.],"y5":[2.]}),
                          {"y1":"y5"}, {"y1":0.5})


class TestDatabase(unittest.TestCase):
    DB = "/tmp/_test_nb.db"

    def setUp(self):
        self.engine = get_engine(self.DB)

    def tearDown(self):
        if os.path.exists(self.DB):
            os.remove(self.DB)

    def test_save_and_reload(self):
        df = pd.DataFrame({"x":[1.,2.],"y":[3.,4.]})
        save_to_db(df, "tbl", self.engine)
        res = load_from_db("tbl", self.engine)
        self.assertEqual(len(res), 2)

    def test_save_empty_raises(self):
        with self.assertRaises(DatabaseError):
            save_to_db(pd.DataFrame(), "empty", self.engine)

    def test_append_mode(self):
        df = pd.DataFrame({"x":[1.],"y":[2.]})
        save_to_db(df, "app", self.engine, if_exists="replace")
        save_to_db(df, "app", self.engine, if_exists="append")
        self.assertEqual(len(load_from_db("app", self.engine)), 2)


class TestCountMappedPoints(unittest.TestCase):

    def test_correct_counts(self):
        df = pd.DataFrame({"chosen_ideal_function":["y41","y41","y42",None,"y41"]})
        res = count_mapped_points(df)
        counts = dict(zip(res["chosen_ideal_function"], res["count"]))
        self.assertEqual(counts["y41"], 3)
        self.assertEqual(counts["y42"], 1)

    def test_missing_col_raises(self):
        with self.assertRaises(MissingColumnError):
            count_mapped_points(pd.DataFrame({"wrong":["a"]}))


# ── Run all tests ──────────────────────────────────────────────────────────────
loader = unittest.TestLoader()
suite  = unittest.TestSuite()
for cls in [TestDataLoading, TestFindBestFit, TestFindMaxDeviations,
            TestMapTestData, TestDatabase, TestCountMappedPoints]:
    suite.addTests(loader.loadTestsFromTestCase(cls))

runner = unittest.TextTestRunner(verbosity=2)
result = runner.run(suite)
if result.wasSuccessful():
    print("\n✅ All tests passed!")
else:
    print(f"\n❌ {len(result.failures)} failure(s), {len(result.errors)} error(s).")

test_file_not_found_raises (__main__.TestDataLoading.test_file_not_found_raises) ... ok
test_missing_required_column_raises (__main__.TestDataLoading.test_missing_required_column_raises) ... ok
test_valid_csv_loads (__main__.TestDataLoading.test_valid_csv_loads) ... 2026-05-12 16:00:37,131 [INFO] Loaded '/tmp/_tc_valid.csv' → 1 rows × 2 cols.
ok
test_missing_x_raises (__main__.TestFindBestFit.test_missing_x_raises) ... ok
test_no_ideal_y_cols_raises (__main__.TestFindBestFit.test_no_ideal_y_cols_raises) ... ok
test_returns_four_keys (__main__.TestFindBestFit.test_returns_four_keys) ... 2026-05-12 16:00:37,139 [INFO] y1 → y10  (MSE=0.001467)
2026-05-12 16:00:37,144 [INFO] y2 → y20  (MSE=0.001532)
2026-05-12 16:00:37,149 [INFO] y3 → y30  (MSE=0.002476)
2026-05-12 16:00:37,155 [INFO] y4 → y40  (MSE=0.002273)
ok
test_values_are_valid_ideal_cols (__main__.TestFindBestFit.test_values_are_valid_ideal_cols) ... 2026-05-12 16:00:37,161 [INFO] y1 → y10  (MSE=0.001467)
2026-05-12 16:00:37,167 [IN


✅ All tests passed!


## 14. Results Summary
The analysis identified the best-fit ideal functions for each training variable
by minimising MSE, then mapped test data points using a √2 × max-deviation threshold.

| Training col | Best-fit ideal | Interpretation |
|---|---|---|
| y1 | y42 | Closest ideal function to training y1 |
| y2 | y41 | Closest ideal function to training y2 |
| y3 | y11 | Closest ideal function to training y3 |
| y4 | y48 | Closest ideal function to training y4 |

All datasets and mapping results are persisted to **`assignment.db`** (SQLite).